# End-to-End Paragraph Reconstruction (M-Parallel MLPs + Aux Loss)
**Project:** WSAI/032 - Semantic Text Reconstruction from Embeddings

**Architecture Update:**
This uses **M completely independent MLPs** (where M is `PREFIX_LEN`). 
- Trained end-to-end with **Cross-Entropy Loss** and an **Auxiliary Cosine Loss** to force the soft prompt to semantically match the real text.

In [1]:
!pip install -q transformers peft accelerate tqdm wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 71.6 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 whic

---
## Section 1: Setup & Imports

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import h5py
import os
import time
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
Memory: 15.6 GB


---
## Section 2: Hyperparameters

In [3]:
H5_PATH = '/kaggle/input/datasets/subhavkumar/new-msmarco-embeddings-pair-dataset/corrected_msmarco_token_sentence_embeddings.h5'

# Fallback path finder
if not os.path.exists(H5_PATH):
    for root, dirs, files in os.walk('/kaggle/input'):
        for fname in files:
            if fname.endswith('.h5'):
                H5_PATH = os.path.join(root, fname)
                break

MODEL_NAME = 'Qwen/Qwen3-0.6B'  

PREFIX_LEN = 32          # Number of M-Parallel MLPs (Virtual Tokens)
MAX_TEXT_LEN = 128       # Max length of actual text tokens to train on
PARAGRAPH_DIM = 1024     # Size of input paragraph embeddings

BATCH_SIZE = 8           
EPOCHS = 10               
LEARNING_RATE = 2e-4
TRAIN_SPLIT = 0.90
PATIENCE = 3

print(f'Using {MODEL_NAME} as the Decoder.')
print(f'Using {PREFIX_LEN} Parallel MLPs to map Paragraph Embedding -> Virtual Tokens.')

Using Qwen/Qwen3-0.6B as the Decoder.
Using 32 Parallel MLPs to map Paragraph Embedding -> Virtual Tokens.


---
## Section 3: Load Tokenizer & Dataset

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

class EndToEndDataset(Dataset):
    def __init__(self, h5_path, max_text_len):
        print('Loading dataset...')
        f = h5py.File(h5_path, 'r')
        num = f['seq_lengths'].shape[0]
        
        self.paragraph_embs = []
        self.input_ids = []
        self.attention_masks = []
        self.texts = []
        
        for i in range(num):
            sent_emb = np.array(f['sentence_embeddings'][i], dtype=np.float32)
            ids = np.array(f['input_ids'][i], dtype=np.int64)
            seq_len = int(f['seq_lengths'][i])
            text = f['texts'][i]
            if isinstance(text, bytes): text = text.decode('utf-8')
            
            actual_len = min(seq_len, max_text_len)
            padded_ids = np.full(max_text_len, tokenizer.pad_token_id, dtype=np.int64)
            padded_ids[:actual_len] = ids[:actual_len]
            
            mask = np.zeros(max_text_len, dtype=np.float32)
            mask[:actual_len] = 1.0
            
            self.paragraph_embs.append(sent_emb)
            self.input_ids.append(padded_ids)
            self.attention_masks.append(mask)
            self.texts.append(text)
            
        f.close()
        self.paragraph_embs = torch.tensor(np.array(self.paragraph_embs))
        self.input_ids = torch.tensor(np.array(self.input_ids))
        self.attention_masks = torch.tensor(np.array(self.attention_masks))
        print(f'Done! Embs={self.paragraph_embs.shape}, IDs={self.input_ids.shape}')
    
    def __len__(self): return len(self.paragraph_embs)
    def __getitem__(self, idx):
        return self.paragraph_embs[idx], self.input_ids[idx], self.attention_masks[idx]

full_dataset = EndToEndDataset(H5_PATH, MAX_TEXT_LEN)
train_size = int(TRAIN_SPLIT * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Loading dataset...
Done! Embs=torch.Size([10000, 1024]), IDs=torch.Size([10000, 128])


---
## Section 4: Architecture (M-Parallel MLPs + Aux Loss)

In [5]:
class SingleTokenMLP(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(paragraph_dim, 2048),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(2048, decoder_hidden_dim)
        )
    def forward(self, x):
        return self.net(x)


class EndToEndInverter(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim, prefix_len, decoder_model):
        super().__init__()
        self.prefix_len = prefix_len
        self.decoder_hidden_dim = decoder_hidden_dim
        
        self.m_parallel_mlps = nn.ModuleList([
            SingleTokenMLP(paragraph_dim, decoder_hidden_dim)
            for _ in range(prefix_len)
        ])
        
        self.decoder = decoder_model
        
    def forward(self, paragraph_embs, text_input_ids, attention_mask):
        batch_size = paragraph_embs.shape[0]
        
        # 1. Feed paragraph embedding into all M MLPs in parallel
        outputs = [mlp(paragraph_embs) for mlp in self.m_parallel_mlps]
        
        # Stack them together -> [batch, prefix_len, hidden_dim]
        prefix_embeds = torch.stack(outputs, dim=1) 
        
        # 2. Get embeddings for actual ground-truth text
        text_embeds = self.decoder.get_input_embeddings()(text_input_ids)  
        
        # 3. Cast to bfloat16 to match the decoder, then concatenate
        prefix_embeds = prefix_embeds.to(text_embeds.dtype)
        inputs_embeds = torch.cat([prefix_embeds, text_embeds], dim=1)  
        
        # 4. Create labels (-100 for prefix tokens so loss ignores them)
        prefix_labels = torch.full((batch_size, self.prefix_len), -100, dtype=torch.long).to(text_input_ids.device)
        labels = torch.cat([prefix_labels, text_input_ids], dim=1) 
        
        # 5. Create Attention Mask 
        prefix_mask = torch.ones((batch_size, self.prefix_len), dtype=attention_mask.dtype).to(attention_mask.device)
        concat_mask = torch.cat([prefix_mask, attention_mask], dim=1) 
        
        # 6. Forward pass through decoder to get LM loss
        outputs = self.decoder(inputs_embeds=inputs_embeds, attention_mask=concat_mask, labels=labels)
        lm_loss = outputs.loss
        
        # 7. AUXILIARY LOSS (Cosine Embedding Loss)
        # We force the mean of the generated soft prompt to match the mean of the real text embeddings.
        mean_prefix = prefix_embeds.mean(dim=1)
        
        # Mean of text tokens (accounting for padding mask)
        mask_expanded = attention_mask.unsqueeze(-1).to(text_embeds.dtype)
        sum_text = (text_embeds * mask_expanded).sum(dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        mean_text = sum_text / sum_mask
        
        # Compute Cosine Embedding Loss
        # Target is 1 (we want them to be as similar as possible in direction)
        target = torch.ones(batch_size, device=text_input_ids.device)
        aux_loss = torch.nn.functional.cosine_embedding_loss(mean_prefix, mean_text, target)
        
        # Combine losses (Weighted equally, can be adjusted)
        total_loss = lm_loss + 1.0 * aux_loss
        
        return total_loss
        
    def generate(self, paragraph_embs, tokenizer, max_new_tokens=128):
        batch_size = paragraph_embs.shape[0]
        
        # Generate virtual tokens through parallel MLPs
        outputs = [mlp(paragraph_embs) for mlp in self.m_parallel_mlps]
        prefix_embeds = torch.stack(outputs, dim=1)
        prefix_embeds = prefix_embeds.to(self.decoder.dtype)
        
        # Auto-regressively generate text
        # Added repetition_penalty=1.2 to fix the looping bug!
        outputs = self.decoder.generate(
            inputs_embeds=prefix_embeds,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,  
            repetition_penalty=1.2,
        )
        return tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [6]:
# Load Decoder (device_map={'': device} forces everything onto GPU 0)
print(f'Loading base decoder {MODEL_NAME}...')
base_decoder = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map={'': device}
)

# Apply LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"]
)
decoder_with_lora = get_peft_model(base_decoder, lora_config)
decoder_hidden_size = base_decoder.config.hidden_size

# Initialize model
model = EndToEndInverter(
    paragraph_dim=PARAGRAPH_DIM,
    decoder_hidden_dim=decoder_hidden_size,
    prefix_len=PREFIX_LEN,
    decoder_model=decoder_with_lora
)
model.m_parallel_mlps.to(device)

model.train()
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nModel initialized!')
print(f'Decoder hidden size: {decoder_hidden_size}')
print(f'Trainable params: {trainable_params:,} ({PREFIX_LEN} MLPs + LoRA Adapters)')

Loading base decoder Qwen/Qwen3-0.6B...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


Model initialized!
Decoder hidden size: 1024
Trainable params: 136,609,792 (32 MLPs + LoRA Adapters)


---
## Section 5: Training Loop (with TQDM)

In [7]:
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'test_loss': []}
best_loss = float('inf')
SAVE_PATH = '/kaggle/working/best_end_to_end_aux_model.pt'

print(f'Training for {EPOCHS} epochs...')
print('='*60)

for epoch in range(EPOCHS):
    t0 = time.time()
    
    # --- TRAIN ---
    model.train()
    train_loss, train_steps = 0, 0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for emb, ids, mask in train_pbar:
        emb, ids, mask = emb.to(device), ids.to(device), mask.to(device)
        
        loss = model(emb, ids, mask)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        train_steps += 1
        train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    # --- TEST ---
    model.eval()
    test_loss, test_steps = 0, 0
    
    test_pbar = tqdm(test_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Test]", leave=False)
    with torch.no_grad():
        for emb, ids, mask in test_pbar:
            emb, ids, mask = emb.to(device), ids.to(device), mask.to(device)
            loss = model(emb, ids, mask)
            
            test_loss += loss.item()
            test_steps += 1
            test_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
    t_loss_avg = train_loss / train_steps
    v_loss_avg = test_loss / test_steps
    history['train_loss'].append(t_loss_avg)
    history['test_loss'].append(v_loss_avg)
    
    if v_loss_avg < best_loss:
        best_loss = v_loss_avg
        # FIXED SAVE: Only save trainable parameters (MLPs + LoRA)
        trainable_weights = {k: v for k, v in model.named_parameters() if v.requires_grad}
        torch.save(trainable_weights, SAVE_PATH)
        flag = ' << BEST'
    else:
        flag = ''
        
    scheduler.step()
    
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | Train Loss: {t_loss_avg:.4f} | Test Loss: {v_loss_avg:.4f} | {time.time()-t0:.1f}s {flag}')

Training for 10 epochs...


Epoch  1/10 | Train Loss: 1.8100 | Test Loss: 1.6836 | 2397.7s  << BEST


Epoch  2/10 | Train Loss: 1.5779 | Test Loss: 1.6201 | 2400.9s  << BEST


Epoch  3/10 | Train Loss: 1.4495 | Test Loss: 1.5935 | 2411.2s  << BEST


Epoch  4/10 | Train Loss: 1.3258 | Test Loss: 1.5891 | 2406.7s  << BEST


...

Epoch  6/10 | Train Loss: 1.0895 | Test Loss: 1.6217 | 2405.7s 


Epoch  7/10 | Train Loss: 0.9760 | Test Loss: 1.6673 | 2408.2s 


Epoch  8/10 | Train Loss: 0.8695 | Test Loss: 1.7168 | 2399.2s 


Epoch  9/10 | Train Loss: 0.7837 | Test Loss: 1.7673 | 2397.2s 


KeyboardInterrupt: 

---
## Section 6: Inference & Evaluation

In [8]:
# Load best model (strict=False because base model weights are not in the .pt file)
model.load_state_dict(torch.load(SAVE_PATH), strict=False)
model.eval()

print('='*80)
print('END-TO-END GENERATION RESULTS')
print('='*80)

np.random.seed(42)
sample_indices = np.random.choice(len(test_dataset), size=5, replace=False)

with torch.no_grad():
    for idx in sample_indices:
        emb, ids, mask = test_dataset[idx]
        original_text = full_dataset.texts[test_dataset.indices[idx]]
        
        emb_batch = emb.unsqueeze(0).to(device)
        generated_text = model.generate(emb_batch, tokenizer, max_new_tokens=64)[0]
        
        print(f'\nSample {idx}:')
        print(f'ORIGINAL : {original_text[:200]}')
        print(f'GENERATED: {generated_text}')
        print('-'*80)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


END-TO-END GENERATION RESULTS

Sample 521:
ORIGINAL : Many Sunrise Senior Living communities have also received the Senior Advisor Excellence Award based on ratings and reviews from consumers. To qualify for an Excellence Award, communities must have rec
GENERATED: The Senior Housing Awards are the highest recognition given to senior housing in our community. The rating is based on a comprehensive review of all 1,208 Star Ratings awarded by experienced and independent reviewers from across the country.
--------------------------------------------------------------------------------

Sample 737:
ORIGINAL : You may order copies of Colorado vital records through VitalChek on an expedited basis. Colorado Vital Records requires all applicants to submit a copy of proof of relationship (e.g. marriage certific
GENERATED: To obtain a copy of your Colorado personal records, you will need to provide the following information: A completed application form (see below) Your birth certificate or marr

In [11]:
# Create a new filename so we don't overwrite your tiny weights file
SAVE_ENTIRE_PATH = '/kaggle/working/new_end_to_end_model.pt'

print("Saving the ENTIRE 2.4 GB model... this might take a minute...")

# This instantly saves the MLPs, the LoRA adapters, and the base Qwen weights 
# exactly as they are right now in memory!
torch.save(model.state_dict(), SAVE_ENTIRE_PATH)

print(f"Done! Entire model successfully saved to: {SAVE_ENTIRE_PATH}")

Saving the ENTIRE 2.4 GB model... this might take a minute...
Done! Entire model successfully saved to: /kaggle/working/new_end_to_end_model.pt
